In [ ]:
# %%
# --- Consolidated imports ---
import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.tracking import MlflowClient

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

In [ ]:
# %%
df = pd.read_csv("../notebook/fraudTrain.csv")

df.head()

In [ ]:
# %%
df.drop(
    columns=["Unnamed: 0", "trans_num", "first", "last", "street", "dob"],
    inplace=True,
    errors="ignore",
)

In [ ]:
# %%
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])

df["hour"] = df["trans_date_trans_time"].dt.hour
df["day"] = df["trans_date_trans_time"].dt.day

df.drop("trans_date_trans_time", axis=1, inplace=True)

In [ ]:
# %%
encoder = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col] = encoder.fit_transform(df[col])

In [ ]:
# %%
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

In [ ]:
# %%
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [ ]:
# %%
print(mlflow.__version__)

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Anomaly Detection MLflow")

print("MLflow Connected")

In [ ]:
# %%
def log_metrics(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True)

    mlflow.log_metric("accuracy", accuracy_score(y_true, y_pred))
    mlflow.log_metric("recall_class_0", report["0"]["recall"])
    mlflow.log_metric("recall_class_1", report["1"]["recall"])
    mlflow.log_metric("f1_score_macro", report["macro avg"]["f1-score"])

In [ ]:
# %%
with mlflow.start_run(run_name="Logistic Regression"):
    model = LogisticRegression(C=1, solver="liblinear", random_state=42)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mlflow.log_param("C", 1)
    mlflow.log_param("solver", "liblinear")
    log_metrics(y_test, predictions)
    mlflow.sklearn.log_model(model, "model")

In [ ]:
# %%
with mlflow.start_run(run_name="Random Forest"):
    model = RandomForestClassifier(n_estimators=30, max_depth=3, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mlflow.log_param("n_estimators", 30)
    mlflow.log_param("max_depth", 3)
    log_metrics(y_test, pred)
    mlflow.sklearn.log_model(model, "model")

In [ ]:
# %%
with mlflow.start_run(run_name="XGBoost"):
    model = XGBClassifier(eval_metric="logloss", random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mlflow.log_param("eval_metric", "logloss")
    log_metrics(y_test, pred)
    mlflow.xgboost.log_model(model, "model")

In [ ]:
# %%
with mlflow.start_run(run_name="XGBoost + SMOTE"):
    model = XGBClassifier(eval_metric="logloss", random_state=42)
    model.fit(X_train_smote, y_train_smote)
    pred = model.predict(X_test)

    mlflow.log_param("eval_metric", "logloss")
    mlflow.log_param("sampling", "SMOTE")
    log_metrics(y_test, pred)
    mlflow.xgboost.log_model(model, "model")

In [ ]:
# %%
# --- Find the best run ---
client = MlflowClient()

experiment = client.get_experiment_by_name("Anomaly Detection MLflow")

runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

runs_df = runs_df.sort_values(
    by=["metrics.recall_class_1", "metrics.f1_score_macro"],
    ascending=False,
)

print(
    runs_df[
        [
            "run_id",
            "tags.mlflow.runName",
            "metrics.accuracy",
            "metrics.recall_class_1",
            "metrics.f1_score_macro",
        ]
    ]
)

In [ ]:
# %%
best_run = runs_df.iloc[0]
best_run_id = best_run["run_id"]
best_model_name = best_run["tags.mlflow.runName"]

print(f"Best run ID   : {best_run_id}")
print(f"Best run name : {best_model_name}")

In [ ]:
# %%
# --- Register the best model, then assign aliases ---
# FIX: build the URI from best_run_id instead of a hardcoded placeholder;
#      create the version first, then set aliases on the version that exists.
model_uri = f"runs:/{best_run_id}/model"

mv = mlflow.register_model(model_uri, "anomaly-detection-prod")

client.set_registered_model_alias(
    name="anomaly-detection-prod",
    alias="challenger",
    version=mv.version,
)

client.set_registered_model_alias(
    name="anomaly-detection-prod",
    alias="champion",
    version=mv.version,
)

print(f"Registered version {mv.version} as champion & challenger")

In [ ]:
# %%
# --- Load the champion model and run inference ---
# FIX: load by alias instead of a hardcoded version number.
model = mlflow.pyfunc.load_model("models:/anomaly-detection-prod@champion")

predictions = model.predict(X_test)

print(predictions[:10])